# Modulo 1 · Lezione 12 — Progetto Integratore
# Sistema di Gestione Biblioteca

**Corso di Intelligenza Artificiale — Livello base**

---

## Benvenuto al progetto integratore

Questa lezione chiude il Modulo 1. Non impareremo più concetti nuovi: metteremo **insieme tutto ciò che sai** per costruire un sistema reale dall'inizio alla fine.

Un progetto integratore è diverso dagli esercizi precedenti:
- Non c'è una soluzione unica né un percorso segnato.
- Dovrai **prendere decisioni di design** e giustificarle.
- L'obiettivo è che alla fine tu sia in grado di leggere il codice e capire **perché** è scritto così, non solo **cosa** fa.

---

## Quali concetti mette in pratica

| Lezione | Concetto | Dove compare in questo progetto |
|-------|----------|--------------------------------|
| L1–L3 | Variabili, tipi, operatori | Ovunque |
| L4–L5 | Controllo di flusso, cicli | Menu, ricerche, filtri |
| L6 | Collezioni | Catalogo, prestiti, storico |
| L7 | Funzioni | Ricerca, report, utilità |
| L8 | File (JSON) | Persistenza del catalogo e dei dati |
| L9 | Eccezioni | Dominio proprio: `MaterialeNonDisponibile`, ecc. |
| L10–L11 | POO (ereditarietà, polimorfismo, ABC) | `Materiale → Libro / Rivista / DVD` |

---

## Il sistema che costruiremo

```
┌─────────────────────────────────────────────────┐
│              BIBLIOTECA DIGITALE                │
├─────────────────┬───────────────────────────────┤
│   CATALOGO      │   UTENTI        │   PRESTITI  │
│  Libro          │   Utente        │   Prestito  │
│  Rivista        │   (nome,        │   (chi,     │
│  DVD            │    email,       │    cosa,    │
│  (ereditarietà  │    storico)     │    quando)  │
│   ABC)          │                 │             │
├─────────────────┴───────────────────────────────┤
│              BIBLIOTECA (orchestratore)         │
│  cerca · presta · restituisce · report          │
├─────────────────────────────────────────────────┤
│              PERSISTENZA (JSON)                 │
│  salva() · carica()                             │
└─────────────────────────────────────────────────┘
```

## Piano di costruzione

| Blocco | Cosa costruisci |
|--------|----------------|
| **1** | Eccezioni e gerarchia di `Materiale` (ABC + sottoclassi) |
| **2** | Classi `Utente` e `Prestito` |
| **3** | Classe `Biblioteca`: catalogo, aggiunta e rimozione dei materiali |
| **4** | Prestiti, restituzioni e ricerca polimorfica |
| **5 (sfida)** | Persistenza JSON, report e menu interattivo |

---
## Blocco 1 — Eccezioni e gerarchia dei materiali

### 1.1 Eccezioni del dominio

Prima di scrivere qualsiasi classe di business, definiamo le eccezioni. Così il resto del codice può lanciarle e catturarle con nomi significativi.

In [ ]:
# ─── Gerarchia di eccezioni della biblioteca ────────────

class ErroreBiblioteca(Exception):
    """Base per tutti gli errori della biblioteca."""

class MaterialeNonTrovato(ErroreBiblioteca):
    """L'ISBN / ID cercato non esiste nel catalogo."""

class MaterialeNonDisponibile(ErroreBiblioteca):
    """Il materiale esiste ma è in prestito."""

class UtenteNonTrovato(ErroreBiblioteca):
    """L'ID utente non esiste."""

class LimitePrestitiRaggiunto(ErroreBiblioteca):
    """L'utente ha già il massimo di prestiti attivi."""

class PrestitoNonTrovato(ErroreBiblioteca):
    """Non c'è nessun prestito attivo per quel materiale e utente."""

class DatiNonValidi(ErroreBiblioteca):
    """Dati di input non corretti o incompleti."""

print("✅ Gerarchia di eccezioni definita")

### 1.2 La gerarchia dei materiali

Tutti i materiali della biblioteca condividono un'interfaccia comune — la classe astratta `Materiale`. Ogni tipo concreto aggiunge i propri attributi e sovrascrive i metodi che lo differenziano.

```
Materiale (ABC)
  id, titolo, disponibile
  presta(), restituisci()   ← concreti nella base
  tipo(), scheda()          ← astratti, ogni sottoclasse li definisce
  to_dict() / from_dict()   ← per serializzare in JSON
       │
       ├── Libro    (autore, isbn, genere, pagine)
       ├── Rivista  (editore, numero, mese_anno)
       └── DVD      (regista, durata_min, classificazione)
```

**Consegna blocco 1:** Implementa la classe `Materiale` e le sue tre sottoclassi. Requisiti:

- `Materiale.__init__` assegna un `id` autoincrementale (`M001`, `M002`…), `titolo` e `disponibile=True`.
- `presta()` imposta `disponibile=False` o lancia `MaterialeNonDisponibile`.
- `restituisci()` imposta `disponibile=True`.
- `@abstractmethod tipo()` → restituisce `"Libro"`, `"Rivista"` o `"DVD"`.
- `@abstractmethod scheda()` → restituisce una stringa con i dettagli del materiale.
- `to_dict()` → serializza in dizionario (per salvare in JSON). Include `"classe": self.tipo()`.
- `@classmethod from_dict(cls, dati)` → ricostruisce l'oggetto da un dizionario.
- `__str__` usa `scheda()`; `__repr__` include `tipo()`, `id` e `titolo`; `__eq__` confronta per `id`.

In [ ]:
# ✏️ IL TUO CODICE QUI
from abc import ABC, abstractmethod


<details>
<summary>💡 Vedi soluzione — Blocco 1</summary>

```python
from abc import ABC, abstractmethod

class Materiale(ABC):
    _contatore = 0

    def __init__(self, titolo):
        Materiale._contatore += 1
        self.id          = f"M{Materiale._contatore:03d}"
        self.titolo      = titolo.strip()
        self.disponibile = True

    # ─── Metodi concreti ───────────────────────────────
    def presta(self):
        if not self.disponibile:
            raise MaterialeNonDisponibile(
                f"'{self.titolo}' [{self.id}] è già in prestito"
            )
        self.disponibile = False

    def restituisci(self):
        self.disponibile = True

    # ─── Metodi astratti ──────────────────────────────
    @abstractmethod
    def tipo(self):
        pass

    @abstractmethod
    def scheda(self):
        pass

    # ─── Serializzazione ───────────────────────────────────
    def to_dict(self):
        return {
            "classe":      self.tipo(),
            "id":          self.id,
            "titolo":      self.titolo,
            "disponibile": self.disponibile,
        }

    @classmethod
    def from_dict(cls, dati):
        """Ricostruisce un Materiale da un dizionario. Da adattare nelle sottoclassi."""
        obj = cls.__new__(cls)          # non chiama __init__
        obj.id          = dati["id"]
        obj.titolo      = dati["titolo"]
        obj.disponibile = dati["disponibile"]
        return obj

    # ─── Dunder ──────────────────────────────────────────
    def __str__(self):
        stato = "✅" if self.disponibile else "🔴"
        return f"{stato} [{self.id}] {self.scheda()}"

    def __repr__(self):
        return f"{self.tipo()}(id={self.id!r}, titolo={self.titolo!r})"

    def __eq__(self, altro):
        if not isinstance(altro, Materiale):
            return NotImplemented
        return self.id == altro.id

    def __hash__(self):
        return hash(self.id)


class Libro(Materiale):
    def __init__(self, titolo, autore, isbn, genere="Generale", pagine=0):
        super().__init__(titolo)
        self.autore = autore.strip()
        self.isbn   = isbn.strip()
        self.genere = genere
        self.pagine = int(pagine)

    def tipo(self):
        return "Libro"

    def scheda(self):
        return (
            f"LIBRO  · {self.titolo} / {self.autore} "
            f"· ISBN {self.isbn} · {self.genere} · {self.pagine} pag."
        )

    def to_dict(self):
        d = super().to_dict()
        d.update({"autore": self.autore, "isbn": self.isbn,
                  "genere": self.genere, "pagine": self.pagine})
        return d

    @classmethod
    def from_dict(cls, dati):
        obj = cls(dati["titolo"], dati["autore"], dati["isbn"],
                  dati.get("genere", "Generale"), dati.get("pagine", 0))
        obj.id          = dati["id"]
        obj.disponibile = dati["disponibile"]
        return obj


class Rivista(Materiale):
    def __init__(self, titolo, editore, numero, mese_anno):
        super().__init__(titolo)
        self.editore   = editore.strip()
        self.numero    = int(numero)
        self.mese_anno = mese_anno.strip()    # es: "Giugno 2025"

    def tipo(self):
        return "Rivista"

    def scheda(self):
        return (
            f"RIVISTA· {self.titolo} · Ed. {self.editore} "
            f"· N. {self.numero} · {self.mese_anno}"
        )

    def to_dict(self):
        d = super().to_dict()
        d.update({"editore": self.editore, "numero": self.numero,
                  "mese_anno": self.mese_anno})
        return d

    @classmethod
    def from_dict(cls, dati):
        obj = cls(dati["titolo"], dati["editore"],
                  dati["numero"], dati["mese_anno"])
        obj.id          = dati["id"]
        obj.disponibile = dati["disponibile"]
        return obj


class DVD(Materiale):
    def __init__(self, titolo, regista, durata_min, classificazione="TP"):
        super().__init__(titolo)
        self.regista          = regista.strip()
        self.durata_min       = int(durata_min)
        self.classificazione  = classificazione.strip().upper()

    def tipo(self):
        return "DVD"

    def scheda(self):
        return (
            f"DVD    · {self.titolo} · Reg. {self.regista} "
            f"· {self.durata_min} min · {self.classificazione}"
        )

    def to_dict(self):
        d = super().to_dict()
        d.update({"regista": self.regista, "durata_min": self.durata_min,
                  "classificazione": self.classificazione})
        return d

    @classmethod
    def from_dict(cls, dati):
        obj = cls(dati["titolo"], dati["regista"],
                  dati["durata_min"], dati.get("classificazione", "TP"))
        obj.id          = dati["id"]
        obj.disponibile = dati["disponibile"]
        return obj


# ─── Prove ─────────────────────────────────────────────
Materiale._contatore = 0

l1 = Libro("Don Chisciotte", "Miguel de Cervantes", "978-84-376-0494-7", "Classico", 863)
l2 = Libro("Dune",           "Frank Herbert",       "978-84-450-7652-0", "Fantascienza", 604)
r1 = Rivista("National Geographic", "National Geographic Society", 312, "Maggio 2025")
d1 = DVD("Blade Runner 2049", "Denis Villeneuve", 164, "16")

for m in [l1, l2, r1, d1]:
    print(m)

print()
l1.presta()
print(f"Dopo il prestito: {l1}")

try:
    l1.presta()
except MaterialeNonDisponibile as e:
    print(f"❌ {e}")

l1.restituisci()
print(f"Dopo la restituzione: {l1}")

print()
print("to_dict:", l1.to_dict())
l1_ricostruito = Libro.from_dict(l1.to_dict())
print("from_dict:", l1_ricostruito)
print(f"Sono uguali: {l1 == l1_ricostruito}")

# Polimorfismo: stesso ciclo, tipi diversi
print("\n─── Tipi polimorfici ───")
catalogo_prova = [l1, l2, r1, d1]
for m in catalogo_prova:
    print(f"  {m.tipo():8} → {m.titolo}")
```

**Decisioni di design da commentare in classe:**
- `__hash__` viene definito insieme a `__eq__` perché i materiali possano essere usati in set e come chiavi di dict.
- `from_dict` usa `cls(...)` nelle sottoclassi e poi corregge `id` e `disponibile` — così l'`__init__` assegnerebbe un nuovo id, ma lo sovrascriviamo con quello salvato. Alternativa: passare `_override_id` come parametro.
- `to_dict` nelle sottoclassi chiama `super().to_dict()` e poi aggiorna — pattern di estensione pulito.
</details>

---
## Blocco 2 — `Utente` e `Prestito`

**Consegna:** Implementa due classi di supporto:

### `Utente`
- `__init__(nome, email, max_prestiti=3)` — id autoincrementale (`U001`…), nome, email, lista di `prestiti_attivi` (id dei materiali).
- `puo_prendere_in_prestito()` → `True` se `len(prestiti_attivi) < max_prestiti`.
- `registra_prestito(id_materiale)` / `registra_restituzione(id_materiale)`.
- `to_dict()` / `@classmethod from_dict(cls, dati)`.
- `__str__`, `__repr__`, `__eq__` per `id`.

### `Prestito`
- `__init__(utente, materiale, giorni_termine=14)` — salva riferimenti, `data_prestito=date.today()`, `data_restituzione=None`.
- `@property data_limite` → `data_prestito + timedelta(giorni_termine)`.
- `@property attivo` → `data_restituzione is None`.
- `@property giorni_rimanenti` → giorni fino a `data_limite` (negativo se scaduto).
- `@property in_mora` → `True` se attivo e `giorni_rimanenti < 0`.
- `chiudi()` → registra `data_restituzione = date.today()`.
- `to_dict()` — salva gli id, non gli oggetti completi.
- `__str__` leggibile con stato visivo (attivo / scaduto / restituito).

In [ ]:
# ✏️ IL TUO CODICE QUI
from datetime import date, timedelta


<details>
<summary>💡 Vedi soluzione — Blocco 2</summary>

```python
from datetime import date, timedelta

class Utente:
    _contatore = 0

    def __init__(self, nome, email, max_prestiti=3):
        Utente._contatore += 1
        self.id              = f"U{Utente._contatore:03d}"
        self.nome            = nome.strip().title()
        self.email           = email.strip().lower()
        self.max_prestiti    = int(max_prestiti)
        self.prestiti_attivi = []   # lista di id_materiale

    def puo_prendere_in_prestito(self):
        return len(self.prestiti_attivi) < self.max_prestiti

    def registra_prestito(self, id_materiale):
        if not self.puo_prendere_in_prestito():
            raise LimitePrestitiRaggiunto(
                f"{self.nome} ha già {len(self.prestiti_attivi)} "
                f"prestito/i attivo/i (massimo: {self.max_prestiti})"
            )
        if id_materiale in self.prestiti_attivi:
            raise ErroreBiblioteca(f"{self.nome} ha già in prestito '{id_materiale}'")
        self.prestiti_attivi.append(id_materiale)

    def registra_restituzione(self, id_materiale):
        if id_materiale not in self.prestiti_attivi:
            raise PrestitoNonTrovato(
                f"{self.nome} non ha nessun prestito attivo per '{id_materiale}'"
            )
        self.prestiti_attivi.remove(id_materiale)

    def to_dict(self):
        return {
            "id":              self.id,
            "nome":            self.nome,
            "email":           self.email,
            "max_prestiti":    self.max_prestiti,
            "prestiti_attivi": self.prestiti_attivi[:],
        }

    @classmethod
    def from_dict(cls, dati):
        obj = cls(dati["nome"], dati["email"], dati.get("max_prestiti", 3))
        obj.id              = dati["id"]
        obj.prestiti_attivi = dati.get("prestiti_attivi", [])
        return obj

    def __str__(self):
        attivi = len(self.prestiti_attivi)
        return (
            f"[{self.id}] {self.nome} ({self.email}) "
            f"— {attivi}/{self.max_prestiti} prestiti"
        )

    def __repr__(self):
        return f"Utente(id={self.id!r}, nome={self.nome!r})"

    def __eq__(self, altro):
        if not isinstance(altro, Utente):
            return NotImplemented
        return self.id == altro.id

    def __hash__(self):
        return hash(self.id)


class Prestito:
    _contatore = 0

    def __init__(self, utente, materiale, giorni_termine=14):
        Prestito._contatore   += 1
        self.id                = f"P{Prestito._contatore:05d}"
        self.utente            = utente
        self.materiale         = materiale
        self.giorni_termine    = int(giorni_termine)
        self.data_prestito     = date.today()
        self.data_restituzione = None

    @property
    def data_limite(self):
        return self.data_prestito + timedelta(days=self.giorni_termine)

    @property
    def attivo(self):
        return self.data_restituzione is None

    @property
    def giorni_rimanenti(self):
        if not self.attivo:
            return 0
        return (self.data_limite - date.today()).days

    @property
    def in_mora(self):
        return self.attivo and self.giorni_rimanenti < 0

    def chiudi(self):
        self.data_restituzione = date.today()

    def to_dict(self):
        return {
            "id":                self.id,
            "id_utente":         self.utente.id,
            "id_materiale":      self.materiale.id,
            "giorni_termine":    self.giorni_termine,
            "data_prestito":     str(self.data_prestito),
            "data_restituzione": str(self.data_restituzione) if self.data_restituzione else None,
        }

    def __str__(self):
        if not self.attivo:
            stato = f"✅ Restituito il {self.data_restituzione}"
        elif self.in_mora:
            stato = f"🔴 SCADUTO ({abs(self.giorni_rimanenti)} giorni di mora)"
        else:
            stato = f"📖 Attivo ({self.giorni_rimanenti} giorni rimanenti)"
        return (
            f"[{self.id}] {self.utente.nome} ← '{self.materiale.titolo}' "
            f"| {self.data_prestito} → {self.data_limite} | {stato}"
        )


# ─── Prove ─────────────────────────────────────────────
Materiale._contatore = 0
Utente._contatore    = 0
Prestito._contatore  = 0

u1 = Utente("anna rossi", "anna@mail.com", max_prestiti=2)
u2 = Utente("Luigi Bianchi", "luigi@mail.com")
print(u1)
print(u2)

l1 = Libro("Dune", "Frank Herbert", "978-84-450-7652-0", "FS", 604)

p = Prestito(u1, l1, giorni_termine=14)
print(f"\n{p}")
print(f"Data limite: {p.data_limite}")
print(f"Giorni rimanenti: {p.giorni_rimanenti}")

u1.registra_prestito(l1.id)
print(f"\nUtente dopo il prestito: {u1}")

# Simulare la mora
p.data_prestito = date.today() - timedelta(days=20)
print(f"\nSimulando mora: {p}")
print(f"In mora: {p.in_mora}")

# Restituire
p.chiudi()
u1.registra_restituzione(l1.id)
print(f"\nDopo la restituzione: {p}")
print(f"Utente: {u1}")
```

**Decisioni di design:**
- `Prestito` salva **riferimenti** all'oggetto `utente` e `materiale`, non solo i loro id — così a runtime possiamo accedere a tutti i loro dati senza ricerche aggiuntive. `to_dict()` salva solo gli id per la serializzazione.
- `giorni_rimanenti` restituisce 0 se il prestito è già chiuso — evita valori negativi confusi in prestiti già restituiti.
- `prestiti_attivi` in `Utente` è una lista di `id_materiale` (stringhe), non di oggetti `Materiale` — più leggero e facile da serializzare.
</details>

---
## Blocco 3 — `Biblioteca`: catalogo e utenti

**Consegna:** Implementa la classe `Biblioteca` con la gestione del catalogo e degli utenti. Ancora non i prestiti (sarà il blocco 4).

**Catalogo:**
- `aggiungi_materiale(materiale)` — lancia `DatiNonValidi` se non è un'istanza di `Materiale`.
- `elimina_materiale(id_materiale)` — lancia `MaterialeNonTrovato` se non esiste o `MaterialeNonDisponibile` se è in prestito.
- `cerca_per_id(id_materiale)` → materiale o lancia `MaterialeNonTrovato`.
- `cerca(termine)` → lista di materiali il cui titolo, autore/regista/editore contiene il termine (case-insensitive).
- `filtra(tipo=None, disponibile=None)` → lista filtrata per tipo (`"Libro"`, `"Rivista"`, `"DVD"`) e/o disponibilità.

**Utenti:**
- `registra_utente(utente)` — lancia `DatiNonValidi` se l'email esiste già.
- `cerca_utente(id_utente)` → utente o lancia `UtenteNonTrovato`.

**Collezioni interne:** usa dizionari `{id: oggetto}` per ricerche `O(1)` per id.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Blocco 3</summary>

```python
class Biblioteca:
    def __init__(self, nome):
        self.nome       = nome
        self._catalogo  = {}   # {id_materiale: Materiale}
        self._utenti    = {}   # {id_utente: Utente}
        self._prestiti  = {}   # {id_prestito: Prestito}  — blocco 4

    # ─── Catalogo ─────────────────────────────────────────
    def aggiungi_materiale(self, materiale):
        if not isinstance(materiale, Materiale):
            raise DatiNonValidi(f"Atteso un Materiale, non {type(materiale).__name__}")
        self._catalogo[materiale.id] = materiale
        print(f"📚 Aggiunto: {materiale}")

    def elimina_materiale(self, id_materiale):
        materiale = self.cerca_per_id(id_materiale)   # lancia MaterialeNonTrovato
        if not materiale.disponibile:
            raise MaterialeNonDisponibile(
                f"Impossibile eliminare '{materiale.titolo}': è in prestito"
            )
        del self._catalogo[id_materiale]
        print(f"🗑️  Eliminato: {materiale.titolo}")
        return materiale

    def cerca_per_id(self, id_materiale):
        materiale = self._catalogo.get(id_materiale)
        if materiale is None:
            raise MaterialeNonTrovato(f"Non esiste nessun materiale con id '{id_materiale}'")
        return materiale

    def cerca(self, termine):
        """Ricerca nel titolo e nei campi di testo di ogni tipo."""
        termine = termine.strip().lower()
        risultati = []
        for m in self._catalogo.values():
            campi = [m.titolo]
            if isinstance(m, Libro):
                campi += [m.autore, m.isbn, m.genere]
            elif isinstance(m, Rivista):
                campi += [m.editore]
            elif isinstance(m, DVD):
                campi += [m.regista, m.classificazione]
            if any(termine in campo.lower() for campo in campi):
                risultati.append(m)
        return risultati

    def filtra(self, tipo=None, disponibile=None):
        """Filtra il catalogo per tipo e/o disponibilità."""
        risultato = list(self._catalogo.values())
        if tipo is not None:
            risultato = [m for m in risultato if m.tipo() == tipo]
        if disponibile is not None:
            risultato = [m for m in risultato if m.disponibile == disponibile]
        return risultato

    # ─── Utenti ─────────────────────────────────────────
    def registra_utente(self, utente):
        if not isinstance(utente, Utente):
            raise DatiNonValidi(f"Atteso un Utente, non {type(utente).__name__}")
        email_esistenti = {u.email for u in self._utenti.values()}
        if utente.email in email_esistenti:
            raise DatiNonValidi(f"L'email '{utente.email}' è già registrata")
        self._utenti[utente.id] = utente
        print(f"👤 Registrato: {utente}")

    def cerca_utente(self, id_utente):
        utente = self._utenti.get(id_utente)
        if utente is None:
            raise UtenteNonTrovato(f"Non esiste nessun utente con id '{id_utente}'")
        return utente

    # ─── Utilità di base ───────────────────────────────
    def __len__(self):
        return len(self._catalogo)

    def __str__(self):
        return (
            f"{self.nome} — {len(self._catalogo)} materiali, "
            f"{len(self._utenti)} utenti, "
            f"{len(self._prestiti)} prestiti attivi"
        )


# ─── Prove ─────────────────────────────────────────────
Materiale._contatore = 0
Utente._contatore    = 0

bib = Biblioteca("Biblioteca Comunale")

libri = [
    Libro("Don Chisciotte",      "Miguel de Cervantes", "978-84-376-0494-7", "Classico",     863),
    Libro("Dune",                "Frank Herbert",       "978-84-450-7652-0", "Fantascienza", 604),
    Libro("Sapiens",             "Yuval Noah Harari",   "978-84-9992-950-3", "Saggistica",   443),
    Libro("Python per tutti",    "Charles Severance",   "978-1-530051-43-7", "Tecnologia",   242),
]
riviste = [
    Rivista("National Geographic", "NGS",       312, "Maggio 2025"),
    Rivista("MIT Tech Review",     "MIT Press",  89, "Giugno 2025"),
]
dvds = [
    DVD("Blade Runner 2049",  "Denis Villeneuve",  164, "16"),
    DVD("Interstellar",       "Christopher Nolan", 169, "TP"),
]

for m in libri + riviste + dvds:
    bib.aggiungi_materiale(m)

for nome, email in [("Anna Rossi", "anna@mail.com"), ("Luigi Bianchi", "luigi@mail.com"),
                     ("Marta Verdi", "marta@mail.com")]:
    bib.registra_utente(Utente(nome, email))

print(f"\n{bib}")

print("\n─── Ricerca: 'dune' ───")
for m in bib.cerca("dune"):
    print(f"  {m}")

print("\n─── Filtro: Libri disponibili ───")
for m in bib.filtra(tipo="Libro", disponibile=True):
    print(f"  {m}")
```

**Decisioni di design:**
- Dizionari interni (`_catalogo`, `_utenti`, `_prestiti`) con id come chiave — ricerca in O(1).
- `cerca` usa `isinstance` all'interno della classe stessa — è accettabile perché `Biblioteca` è il livello che conosce tutte le sottoclassi.
- `{u.email for u in self._utenti.values()}` — set comprehension per il controllo dei duplicati in O(1).
</details>

---
## Blocco 4 — Prestiti, restituzioni e ricerca polimorfica

**Consegna:** Completa la classe `Biblioteca` aggiungendo la gestione dei prestiti:

- **`presta(id_utente, id_materiale, giorni_termine=14)`**:
  1. Cerca l'utente e il materiale (lancia se non esistono).
  2. Verifica che l'utente possa prendere in prestito.
  3. Chiama `materiale.presta()` (polimorfismo — funziona per Libro, Rivista o DVD).
  4. Chiama `utente.registra_prestito(id_materiale)`.
  5. Crea il `Prestito`, lo salva e lo restituisce.

- **`restituisci(id_utente, id_materiale)`**:
  1. Trova il prestito attivo corrispondente.
  2. Chiude il prestito, restituisce il materiale e aggiorna l'utente.

- **`prestiti_attivi()`** → lista di prestiti senza `data_restituzione`.
- **`prestiti_in_mora()`** → lista di prestiti attivi con `in_mora == True`.
- **`storico_utente(id_utente)`** → tutti i prestiti di quell'utente.
- **`elenca_catalogo()`** → stampa il catalogo completo organizzato per tipo.

In [ ]:
# ✏️ IL TUO CODICE QUI — aggiungere metodi alla classe Biblioteca


<details>
<summary>💡 Vedi soluzione — Blocco 4</summary>

```python
# Ridefinire Biblioteca completa con tutti i blocchi precedenti più i nuovi metodi

class Biblioteca:
    def __init__(self, nome):
        self.nome      = nome
        self._catalogo = {}
        self._utenti   = {}
        self._prestiti = {}

    # ── Catalogo (Blocco 3) ───────────────────────────────
    def aggiungi_materiale(self, materiale):
        if not isinstance(materiale, Materiale):
            raise DatiNonValidi(f"Atteso un Materiale")
        self._catalogo[materiale.id] = materiale

    def elimina_materiale(self, id_materiale):
        materiale = self.cerca_per_id(id_materiale)
        if not materiale.disponibile:
            raise MaterialeNonDisponibile(f"Impossibile eliminare '{materiale.titolo}': è in prestito")
        del self._catalogo[id_materiale]
        return materiale

    def cerca_per_id(self, id_materiale):
        m = self._catalogo.get(id_materiale)
        if m is None:
            raise MaterialeNonTrovato(f"Materiale '{id_materiale}' non trovato")
        return m

    def cerca(self, termine):
        termine = termine.strip().lower()
        risultati = []
        for m in self._catalogo.values():
            campi = [m.titolo]
            if isinstance(m, Libro):     campi += [m.autore, m.isbn, m.genere]
            elif isinstance(m, Rivista): campi += [m.editore]
            elif isinstance(m, DVD):     campi += [m.regista]
            if any(termine in c.lower() for c in campi):
                risultati.append(m)
        return risultati

    def filtra(self, tipo=None, disponibile=None):
        r = list(self._catalogo.values())
        if tipo        is not None: r = [m for m in r if m.tipo() == tipo]
        if disponibile is not None: r = [m for m in r if m.disponibile == disponibile]
        return r

    # ── Utenti (Blocco 3) ───────────────────────────────
    def registra_utente(self, utente):
        if not isinstance(utente, Utente):
            raise DatiNonValidi("Atteso un Utente")
        if any(u.email == utente.email for u in self._utenti.values()):
            raise DatiNonValidi(f"L'email '{utente.email}' è già registrata")
        self._utenti[utente.id] = utente

    def cerca_utente(self, id_utente):
        u = self._utenti.get(id_utente)
        if u is None:
            raise UtenteNonTrovato(f"Utente '{id_utente}' non trovato")
        return u

    # ── Prestiti (Blocco 4) ──────────────────────────────
    def presta(self, id_utente, id_materiale, giorni_termine=14):
        utente    = self.cerca_utente(id_utente)
        materiale = self.cerca_per_id(id_materiale)

        if not utente.puo_prendere_in_prestito():
            raise LimitePrestitiRaggiunto(
                f"{utente.nome} ha raggiunto il limite di {utente.max_prestiti} prestiti"
            )

        materiale.presta()                         # polimorfismo — funziona uguale per Libro, Rivista, DVD
        utente.registra_prestito(materiale.id)

        prestito = Prestito(utente, materiale, giorni_termine)
        self._prestiti[prestito.id] = prestito

        print(f"📖 Prestito registrato: {prestito}")
        return prestito

    def restituisci(self, id_utente, id_materiale):
        utente    = self.cerca_utente(id_utente)
        materiale = self.cerca_per_id(id_materiale)

        # Cercare il prestito attivo corrispondente
        prestito = None
        for p in self._prestiti.values():
            if p.attivo and p.utente.id == id_utente and p.materiale.id == id_materiale:
                prestito = p
                break

        if prestito is None:
            raise PrestitoNonTrovato(
                f"Nessun prestito attivo di '{materiale.titolo}' per {utente.nome}"
            )

        prestito.chiudi()
        materiale.restituisci()
        utente.registra_restituzione(materiale.id)

        mora = "⚠️  (con mora)" if prestito.giorni_rimanenti < 0 else ""
        print(f"✅ Restituzione: '{materiale.titolo}' restituito da {utente.nome} {mora}")
        return prestito

    def prestiti_attivi(self):
        return [p for p in self._prestiti.values() if p.attivo]

    def prestiti_in_mora(self):
        return [p for p in self._prestiti.values() if p.in_mora]

    def storico_utente(self, id_utente):
        utente = self.cerca_utente(id_utente)
        return [p for p in self._prestiti.values() if p.utente.id == id_utente]

    def elenca_catalogo(self):
        from collections import defaultdict
        gruppi = defaultdict(list)
        for m in self._catalogo.values():
            gruppi[m.tipo()].append(m)

        print(f"\n{'═'*70}")
        print(f"  CATALOGO — {self.nome} ({len(self._catalogo)} materiali)")
        print(f"{'═'*70}")
        for tipo in sorted(gruppi):
            gruppo = gruppi[tipo]
            disp   = sum(1 for m in gruppo if m.disponibile)
            print(f"\n  {tipo.upper()} ({len(gruppo)} — {disp} disponibili)")
            print(f"  {'─'*68}")
            for m in gruppo:
                print(f"  {m}")
        print(f"{'═'*70}")

    # ── Dunder ────────────────────────────────────────────
    def __len__(self):
        return len(self._catalogo)

    def __str__(self):
        attivi = len(self.prestiti_attivi())
        return (
            f"{self.nome} — {len(self._catalogo)} materiali, "
            f"{len(self._utenti)} utenti, {attivi} prestiti attivi"
        )


# ─── Scenario di prova completo ────────────────────────
Materiale._contatore = 0
Utente._contatore    = 0
Prestito._contatore  = 0

bib = Biblioteca("Biblioteca Comunale")

for m in [
    Libro("Don Chisciotte",   "Miguel de Cervantes", "978-84-376-0494-7", "Classico",     863),
    Libro("Dune",             "Frank Herbert",       "978-84-450-7652-0", "Fantascienza", 604),
    Libro("Sapiens",          "Yuval Noah Harari",   "978-84-9992-950-3", "Saggistica",   443),
    Libro("Python per tutti", "Charles Severance",   "978-1-530051-43-7", "Tecnologia",   242),
    Rivista("National Geographic", "NGS",       312, "Maggio 2025"),
    Rivista("MIT Tech Review",     "MIT Press",  89, "Giugno 2025"),
    DVD("Blade Runner 2049", "Denis Villeneuve",  164, "16"),
    DVD("Interstellar",      "Christopher Nolan", 169, "TP"),
]:
    bib.aggiungi_materiale(m)

u1 = Utente("Anna Rossi",   "anna@mail.com",  max_prestiti=3)
u2 = Utente("Luigi Bianchi", "luigi@mail.com")
u3 = Utente("Marta Verdi",   "marta@mail.com")
for u in [u1, u2, u3]:
    bib.registra_utente(u)

bib.elenca_catalogo()

print("\n─── Prestiti ───")
p1 = bib.presta(u1.id, "M001", giorni_termine=14)
p2 = bib.presta(u1.id, "M005", giorni_termine=7)
p3 = bib.presta(u2.id, "M007", giorni_termine=3)

# Simulare la mora in p3
p3.data_prestito = date.today() - timedelta(days=10)

print("\n─── Attivi ───")
for p in bib.prestiti_attivi():
    print(f"  {p}")

print("\n─── In mora ───")
for p in bib.prestiti_in_mora():
    print(f"  {p}")

print("\n─── Restituzione ───")
bib.restituisci(u1.id, "M001")

print("\n─── Storico di Anna ───")
for p in bib.storico_utente(u1.id):
    print(f"  {p}")
```

**Decisioni di design:**
- `materiale.presta()` in `Biblioteca.presta` — polimorfismo puro: se domani aggiungiamo `Audiolibro(Materiale)`, il metodo `presta` di `Biblioteca` non cambia.
- Cercare il prestito attivo con un ciclo è O(n) — per una biblioteca piccola va bene. In produzione, aggiungeremmo un indice secondario `{(id_utente, id_materiale): id_prestito}`.
- `prestito.chiudi()` + `materiale.restituisci()` + `utente.registra_restituzione()` — tre operazioni atomiche. Se una fallisce a metà, lo stato resta incoerente. In un sistema reale useremmo transazioni.
</details>

---
## Blocco 5 (sfida) — Persistenza, report e menu interattivo

**Consegna:** Completa il sistema con tre moduli finali:

### 5.1 Persistenza JSON
Aggiungi a `Biblioteca`:
- `salva(percorso)` — serializza catalogo, utenti e prestiti in un file JSON.
- `@classmethod carica(cls, percorso)` — ricostruisce la `Biblioteca` dal JSON.

Per ricostruire i materiali dal JSON serve un **registro delle classi** (`CLASSI_MATERIALE = {"Libro": Libro, "Rivista": Rivista, "DVD": DVD}`). Usalo in `carica`.

### 5.2 Report
Aggiungi a `Biblioteca`:
- `report_generale()` — statistiche globali: totale materiali per tipo, % di disponibilità, top 3 materiali più richiesti (per storico), utente con più prestiti attivi.

### 5.3 Menu interattivo
Scrivi una funzione `menu(biblioteca)` che mostri un menu in ciclo `while True` con opzioni:
```
1. Cerca materiale
2. Prendi in prestito
3. Restituisci
4. Visualizza catalogo
5. Visualizza prestiti attivi
6. Report generale
7. Salva ed esci
```
Usa `try/except ErroreBiblioteca` per catturare errori senza interrompere il ciclo.

In [ ]:
# ✏️ IL TUO CODICE QUI
import json
from pathlib import Path
from collections import Counter


<details>
<summary>💡 Vedi soluzione — Blocco 5</summary>

```python
import json
from pathlib import Path
from collections import Counter

# Registro delle classi per la deserializzazione
CLASSI_MATERIALE = {
    "Libro":   Libro,
    "Rivista": Rivista,
    "DVD":     DVD,
}


# ── 5.1 Persistenza ──────────────────────────────────────

def salva(self, percorso):
    """Serializza tutto lo stato della biblioteca in JSON."""
    dati = {
        "nome":       self.nome,
        "catalogo":   [m.to_dict() for m in self._catalogo.values()],
        "utenti":     [u.to_dict() for u in self._utenti.values()],
        "prestiti":   [p.to_dict() for p in self._prestiti.values()],
        "_contatori": {
            "materiale": Materiale._contatore,
            "utente":    Utente._contatore,
            "prestito":  Prestito._contatore,
        }
    }
    Path(percorso).write_text(
        json.dumps(dati, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    print(f"💾 Biblioteca salvata in '{percorso}'")


@classmethod
def carica(cls, percorso):
    """Ricostruisce la Biblioteca da un JSON."""
    percorso_path = Path(percorso)
    if not percorso_path.exists():
        raise FileNotFoundError(f"Non si trova il file '{percorso}'")

    dati = json.loads(percorso_path.read_text(encoding="utf-8"))

    # Ripristinare i contatori
    contatori = dati.get("_contatori", {})
    Materiale._contatore = contatori.get("materiale", 0)
    Utente._contatore    = contatori.get("utente",    0)
    Prestito._contatore  = contatori.get("prestito",  0)

    bib = cls(dati["nome"])

    # Ricostruire il catalogo
    for d in dati["catalogo"]:
        ClasseM = CLASSI_MATERIALE.get(d["classe"])
        if ClasseM:
            bib._catalogo[d["id"]] = ClasseM.from_dict(d)

    # Ricostruire gli utenti
    for d in dati["utenti"]:
        bib._utenti[d["id"]] = Utente.from_dict(d)

    # Ricostruire i prestiti (servono riferimenti agli oggetti già caricati)
    for d in dati["prestiti"]:
        utente    = bib._utenti.get(d["id_utente"])
        materiale = bib._catalogo.get(d["id_materiale"])
        if utente and materiale:
            p = Prestito.__new__(Prestito)
            p.id                = d["id"]
            p.utente            = utente
            p.materiale         = materiale
            p.giorni_termine    = d["giorni_termine"]
            p.data_prestito     = date.fromisoformat(d["data_prestito"])
            p.data_restituzione = (
                date.fromisoformat(d["data_restituzione"])
                if d["data_restituzione"] else None
            )
            bib._prestiti[p.id] = p

    print(f"📂 Biblioteca caricata da '{percorso}'")
    return bib


# ── 5.2 Report ──────────────────────────────────────────

def report_generale(self):
    from collections import Counter, defaultdict

    print(f"\n{'═'*70}")
    print(f"  REPORT GENERALE — {self.nome}")
    print(f"{'═'*70}")

    # Materiali per tipo
    gruppi = defaultdict(list)
    for m in self._catalogo.values():
        gruppi[m.tipo()].append(m)

    print("\n  CATALOGO:")
    totale = len(self._catalogo)
    for tipo, gruppo in sorted(gruppi.items()):
        disp = sum(1 for m in gruppo if m.disponibile)
        pct  = (disp / len(gruppo) * 100) if gruppo else 0
        print(f"    {tipo:<12} {len(gruppo):>3} unità   disponibili: {disp} ({pct:.0f}%)")
    print(f"    {'TOTALE':<12} {totale:>3} unità")

    # Top 3 materiali più prestati
    conteggio = Counter(p.materiale.id for p in self._prestiti.values())
    print("\n  TOP 3 MATERIALI PIÙ PRESTATI:")
    for id_m, volte in conteggio.most_common(3):
        m = self._catalogo.get(id_m)
        if m:
            print(f"    {volte:>2}x — {m.titolo} ({m.tipo()})")

    # Utenti
    print("\n  UTENTI:")
    print(f"    Registrati: {len(self._utenti)}")
    attivi_per_utente = {u.id: len(u.prestiti_attivi) for u in self._utenti.values()}
    if attivi_per_utente:
        top_id = max(attivi_per_utente, key=attivi_per_utente.get)
        top_u  = self._utenti[top_id]
        print(f"    Con più prestiti attivi: {top_u.nome} ({attivi_per_utente[top_id]})")

    # Mora
    mora = self.prestiti_in_mora()
    print(f"\n  PRESTITI:")
    print(f"    Totale registrati: {len(self._prestiti)}")
    print(f"    Attivi:            {len(self.prestiti_attivi())}")
    print(f"    In mora:           {len(mora)}")
    if mora:
        for p in mora:
            print(f"      ⚠️  {p}")

    print(f"{'═'*70}")


# ── 5.3 Menu interattivo ──────────────────────────────────

def menu(biblioteca):
    """Menu interattivo della biblioteca."""
    OPZIONI = {
        "1": "Cerca materiale",
        "2": "Prendi in prestito",
        "3": "Restituisci",
        "4": "Visualizza catalogo",
        "5": "Visualizza prestiti attivi",
        "6": "Report generale",
        "7": "Salva ed esci",
    }

    while True:
        print(f"\n{'─'*40}")
        print(f"  {biblioteca.nome}")
        print(f"{'─'*40}")
        for chiave, desc in OPZIONI.items():
            print(f"  {chiave}. {desc}")
        print(f"{'─'*40}")

        opzione = input("Scegli un'opzione: ").strip()

        try:
            if opzione == "1":
                termine = input("Termine di ricerca: ").strip()
                risultati = biblioteca.cerca(termine)
                if risultati:
                    for m in risultati: print(f"  {m}")
                else:
                    print("  Nessun risultato.")

            elif opzione == "2":
                id_u = input("ID utente: ").strip()
                id_m = input("ID materiale: ").strip()
                biblioteca.presta(id_u, id_m)

            elif opzione == "3":
                id_u = input("ID utente: ").strip()
                id_m = input("ID materiale: ").strip()
                biblioteca.restituisci(id_u, id_m)

            elif opzione == "4":
                biblioteca.elenca_catalogo()

            elif opzione == "5":
                attivi = biblioteca.prestiti_attivi()
                if attivi:
                    for p in attivi: print(f"  {p}")
                else:
                    print("  Nessun prestito attivo.")

            elif opzione == "6":
                biblioteca.report_generale()

            elif opzione == "7":
                percorso = input("Percorso del file (Invio = biblioteca.json): ").strip()
                if not percorso: percorso = "biblioteca.json"
                biblioteca.salva(percorso)
                print("A presto.")
                break

            else:
                print("Opzione non valida. Scegli tra 1 e 7.")

        except ErroreBiblioteca as e:
            print(f"⚠️  {type(e).__name__}: {e}")
        except KeyboardInterrupt:
            print("\nUscita senza salvare.")
            break


# Iniettare i metodi nella classe (monkey-patching — per questa demo)
Biblioteca.salva           = salva
Biblioteca.carica          = carica
Biblioteca.report_generale = report_generale


# ── Prova di persistenza ────────────────────────────────
# (Usa lo stato del blocco precedente)
bib.salva("biblioteca.json")

Materiale._contatore = 0
Utente._contatore    = 0
Prestito._contatore  = 0

bib2 = Biblioteca.carica("biblioteca.json")
print(bib2)

bib2.report_generale()

# Decommenta per provare il menu in modalità interattiva:
# menu(bib2)
```

**Decisioni di design:**
- `CLASSI_MATERIALE` è il pattern **registro** — caricando il JSON guardiamo la chiave `"classe"` e la usiamo per sapere quale `from_dict` chiamare. Aggiungere un nuovo tipo richiede solo di aggiungere una voce al dizionario.
- I contatori vengono salvati nel JSON per evitare collisioni di id al ricaricamento. Senza questo, caricare e creare nuovi materiali potrebbe generare id duplicati.
- Il menu cattura `ErroreBiblioteca` ma lascia passare `KeyboardInterrupt` — così Ctrl+C esce in modo pulito.
- Il `monkey-patching` (iniettare metodi con `Biblioteca.salva = salva`) serve solo per questa demo del notebook. In produzione tutti i metodi andrebbero dentro la definizione della classe.
</details>

---
## Riflessione finale — Decisioni di design del progetto

Ora che il sistema è completo, ripassa queste domande. Non ci sono risposte uniche — l'obiettivo è che tu possa **argomentare** la tua scelta.

### Domande per il dibattito in classe

**1. Ereditarietà vs composizione**
> `Rivista` eredita da `Materiale`. Ma avrebbe più senso che `Materiale` avesse un attributo `tipo_info` che fosse un oggetto separato?

**2. Dove mettere la logica**
> `materiale.presta()` è in `Materiale`. `Biblioteca.presta()` orchestra l'operazione completa. Perché non mettere tutto in `Materiale`? Perché non mettere tutto in `Biblioteca`?

**3. Relazioni tra oggetti**
> `Prestito` salva riferimenti a `Utente` e `Materiale` in memoria, ma solo id in JSON. Quando conviene salvare l'oggetto completo e quando solo l'id?

**4. Coerenza dello stato**
> In `restituisci()` facciamo tre operazioni: `prestito.chiudi()`, `materiale.restituisci()`, `utente.registra_restituzione()`. Se la seconda fallisce, lo stato resta a metà aggiornato. Come lo risolveresti?

**5. Scalare il sistema**
> Se la biblioteca ha 50.000 materiali, cercare il prestito attivo con un ciclo è troppo lento. Che struttura dati aggiungeresti?

---

## Estensioni opzionali

Se vuoi continuare a esercitarti, ecco alcune idee ordinate dalla più semplice alla più difficile:

| # | Estensione | Concetti che rafforza |
|---|-----------|------------------------|
| A | Aggiungere `Audiolibro(Materiale)` con `narratore` e `durata_min` | Ereditarietà, `from_dict` |
| B | `prenota(id_utente, id_materiale)` — coda d'attesa se è in prestito | Collezioni (deque), POO |
| C | Calcolare la multa per mora (€/giorno) e aggiungerla al report | Proprietà, aritmetica delle date |
| D | Esportare il report in `.txt` e il catalogo in `.csv` | File (L8) |
| E | Validare il formato dell'email con espressioni regolari (`re`) | Modulo `re` |
| F | Test unitari con `unittest` per `presta` e `restituisci` | Testing di base |

---

## Chiusura del Modulo 1

### Cosa hai costruito in questo progetto

```
Gerarchia di eccezioni proprie
Classe astratta Materiale con @abstractmethod
Tre sottoclassi concrete (Libro, Rivista, DVD)
Serializzazione to_dict / from_dict a ogni livello
Classi di supporto (Utente, Prestito) con proprietà calcolate
Orchestratore Biblioteca con ricerche, filtri e statistiche
Persistenza completa in JSON (salvare e caricare)
Menu interattivo con gestione robusta degli errori
```

### Cosa hai applicato dal Modulo 1

```
Variabili · Tipi · Operatori          → Ovunque
Controllo di flusso · Cicli           → Menu, ricerche, filtri
Liste · Dizionari · Set               → Catalogo, utenti, storico
Funzioni · *args · **kwargs           → Metodi di ricerca, report
File (JSON)                           → salva() / carica()
Eccezioni personalizzate              → Dominio bibliotecario completo
POO: ABC, ereditarietà, polimorfismo  → Tutta la gerarchia Materiale
POO: incapsulamento, @property        → Saldo, date, stati
```

---

## Cosa vedremo nel Modulo 2?

Il **Modulo 2** fa il salto da Python puro agli strumenti che usano i professionisti di dati e IA:

| Lezione | Tema |
|-------|------|
| M2 L1 | NumPy — operazioni vettoriali e matrici |
| M2 L2 | Pandas — DataFrame e analisi dati |
| M2 L3 | Visualizzazione con Matplotlib e Seaborn |
| M2 L4 | Introduzione al Machine Learning con scikit-learn |

Tutto ciò che hai imparato nel Modulo 1 è la base. Da qui in poi, userai classi, eccezioni, file e collezioni continuamente — ma su librerie costruite da altri.